In [ ]:
# MNIST Dataset Download and Preparation

This notebook downloads the MNIST dataset and prepares it for the Barren Plateau research project.

**Objective:**
- Download MNIST dataset using TensorFlow Datasets
- Filter for binary classification (digits 3 vs 6)
- Save to appropriate directory structure
- Verify data integrity

In [ ]:
# Import required libraries
import tensorflow as tf
import tensorflow_datasets as tfds
import numpy as np
import matplotlib.pyplot as plt
import os
from pathlib import Path

print(f"TensorFlow version: {tf.__version__}")
print(f"TensorFlow Datasets version: {tfds.__version__}")

## Step 1: Set up directory structure

Create the data directory if it doesn't exist.

In [ ]:
# Define project root and data directory
project_root = Path('/mnt/d/Programs/PF/Hybrid-QNNs')
data_dir = project_root / 'data' / 'mnist'
processed_dir = project_root / 'data' / 'processed'

# Create directories if they don't exist
data_dir.mkdir(parents=True, exist_ok=True)
processed_dir.mkdir(parents=True, exist_ok=True)

print(f"✓ MNIST data directory: {data_dir}")
print(f"✓ Processed data directory: {processed_dir}")
print(f"✓ Directories created successfully!")


## Step 2: Download MNIST Dataset

Download the full MNIST dataset using TensorFlow Datasets and save it to our data directory.

In [ ]:
# Download MNIST dataset
print("Downloading MNIST dataset...")
print("This may take a few minutes on first run...")

# Load MNIST dataset with split information
(ds_train, ds_test), ds_info = tfds.load(
    'mnist',
    split=['train', 'test'],
    shuffle_files=True,
    as_supervised=True,
    with_info=True,
    data_dir=str(data_dir)
)

print(f"\n✓ Dataset downloaded successfully!")
print(f"✓ Dataset location: {data_dir / 'mnist'}")
print(f"\nDataset Info:")
print(f"  - Training samples: {ds_info.splits['train'].num_examples}")
print(f"  - Test samples: {ds_info.splits['test'].num_examples}")
print(f"  - Image shape: {ds_info.features['image'].shape}")
print(f"  - Number of classes: {ds_info.features['label'].num_classes}")

## Step 3: Filter for Binary Classification (3 vs 6)

Extract only digits 3 and 6 for our binary classification task.

In [ ]:
# Filter function for digits 3 and 6
def filter_3_and_6(image, label):
    return tf.logical_or(tf.equal(label, 3), tf.equal(label, 6))

# Apply filter to train and test sets
ds_train_filtered = ds_train.filter(filter_3_and_6)
ds_test_filtered = ds_test.filter(filter_3_and_6)

# Convert to numpy arrays for easier processing
print("Converting to numpy arrays...")
X_train = []
y_train = []
X_test = []
y_test = []

for image, label in ds_train_filtered:
    X_train.append(image.numpy())
    y_train.append(label.numpy())

for image, label in ds_test_filtered:
    X_test.append(image.numpy())
    y_test.append(label.numpy())

X_train = np.array(X_train)
y_train = np.array(y_train)
X_test = np.array(X_test)
y_test = np.array(y_test)

print(f"\n✓ Filtered dataset created!")
print(f"  - Training samples (3 & 6): {len(X_train)}")
print(f"  - Test samples (3 & 6): {len(X_test)}")
print(f"  - Image shape: {X_train.shape[1:]}")
print(f"  - Label distribution (train): 3s={np.sum(y_train==3)}, 6s={np.sum(y_train==6)}")
print(f"  - Label distribution (test): 3s={np.sum(y_test==3)}, 6s={np.sum(y_test==6)}")

## Step 4: Visualize Sample Images

Let's verify the data by visualizing some samples from each class.

In [ ]:
# Visualize sample images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
fig.suptitle('Sample MNIST Images: Digits 3 and 6', fontsize=16, fontweight='bold')

# Show 5 examples of digit 3
indices_3 = np.where(y_train == 3)[0][:5]
for i, ax in enumerate(axes[0]):
    ax.imshow(X_train[indices_3[i]].squeeze(), cmap='gray')
    ax.set_title(f'Label: 3', fontsize=12)
    ax.axis('off')

# Show 5 examples of digit 6
indices_6 = np.where(y_train == 6)[0][:5]
for i, ax in enumerate(axes[1]):
    ax.imshow(X_train[indices_6[i]].squeeze(), cmap='gray')
    ax.set_title(f'Label: 6', fontsize=12)
    ax.axis('off')

plt.tight_layout()
plt.show()

print("✓ Sample images displayed successfully!")

## Step 5: Save Processed Data

Save the filtered dataset to the processed directory for future use.

In [ ]:
# Save the filtered dataset
print("Saving filtered dataset...")

np.save(processed_dir / 'X_train_3_6.npy', X_train)
np.save(processed_dir / 'y_train_3_6.npy', y_train)
np.save(processed_dir / 'X_test_3_6.npy', X_test)
np.save(processed_dir / 'y_test_3_6.npy', y_test)

print(f"\n✓ Dataset saved successfully!")
print(f"  - X_train_3_6.npy: {X_train.shape} ({X_train.nbytes / 1e6:.2f} MB)")
print(f"  - y_train_3_6.npy: {y_train.shape}")
print(f"  - X_test_3_6.npy: {X_test.shape} ({X_test.nbytes / 1e6:.2f} MB)")
print(f"  - y_test_3_6.npy: {y_test.shape}")
print(f"\n✓ Location: {processed_dir}")

## Step 6: Create Dataset Summary

Generate a summary file with dataset statistics.

In [ ]:
# Create dataset summary
summary = f"""
MNIST Binary Classification Dataset Summary
============================================
Date Created: {np.datetime64('today')}
Task: Binary classification (digit 3 vs digit 6)

Dataset Statistics:
-------------------
Training Set:
  - Total samples: {len(X_train)}
  - Digit 3 samples: {np.sum(y_train == 3)}
  - Digit 6 samples: {np.sum(y_train == 6)}
  - Class balance: {np.sum(y_train == 3) / len(y_train) * 100:.1f}% (3) / {np.sum(y_train == 6) / len(y_train) * 100:.1f}% (6)

Test Set:
  - Total samples: {len(X_test)}
  - Digit 3 samples: {np.sum(y_test == 3)}
  - Digit 6 samples: {np.sum(y_test == 6)}
  - Class balance: {np.sum(y_test == 3) / len(y_test) * 100:.1f}% (3) / {np.sum(y_test == 6) / len(y_test) * 100:.1f}% (6)

Image Properties:
-----------------
  - Original shape: 28 x 28 pixels
  - Data type: {X_train.dtype}
  - Value range: [{X_train.min()}, {X_train.max()}]
  - Mean pixel value: {X_train.mean():.2f}
  - Std pixel value: {X_train.std():.2f}

Files Saved:
------------
  - X_train_3_6.npy
  - y_train_3_6.npy
  - X_test_3_6.npy
  - y_test_3_6.npy

Usage for QNN Training:
-----------------------
  - Recommended: Extract 4 principal features for quantum representation
  - Normalize to [0, 1] range (already done)
  - Use 1000 train + 200 test samples for experiments
"""

# Save summary to file
with open(processed_dir / 'dataset_summary.txt', 'w') as f:
    f.write(summary)

print(summary)
print(f"\n✓ Summary saved to: {processed_dir / 'dataset_summary.txt'}")

## ✅ Dataset Download Complete!

The MNIST dataset has been successfully downloaded and processed. You can now use it for training your quantum neural networks.

**Next Steps:**
1. Test the `mnist_loader.py` module in `src/data/`
2. Verify the data can be loaded and reduced to 4 principal components
3. Begin implementing the baseline trainer